教学演示 长期记忆全流程

核心学习目标

掌握长期记忆、短期记忆，理解偏好存储、语义检索与上下文动态注入机制

设计

展示使用向量实现 完整的记忆抽取-合并-存储-检索-使用


| 步骤 | 学什么 | 看哪段源码 / 调用 |
|------|--------|-------------------|
| 源码 A | BM25 + 记忆抽取/合并 | `HybridRetriever`、`MemoryCompressor` |
| 源码 B | 存进 Chroma、检索、拼 Prompt | `SelfBuiltLongTermMemoryImproved` |
|模块 1～5 | 动手跑一遍 | 调用上面的类 |
| 模块 | 本课做什么 | 对应代码 |
| **1. 抽取** | LLM 把自然语言压成 `summary / key_points / importance` | `MemoryCompressor.extract_memory` |
| **2. 合并** | 两条相似记忆合成一条 | `MemoryCompressor.merge_memories` |
| **3. 存储** | 写入管道：去重 → 合并 → Chroma 索引 | `ingest_long_term` / `MemoryWritePipeline` |
| **4. 检索** | 向量 + jieba BM25 + RRF×向量相似度 + 时间/重要性乘性重排 | `search_memories` |
| **5. 使用** | 短期 Checkpoint 模拟 + 长期记忆拼进 Prompt（可选调 LLM） | `build_prompt` + `ChatOpenAI` |

向量默认走 **阿里云 DashScope**（`.env` 里配 `DASHSCOPE_API_KEY`）。


## 0. 环境

```bash
pip install langchain-core langchain-openai chromadb rank-bm25 jieba openai python-dotenv httpx numpy ipywidgets
```

在项目根目录放 `.env`（只写 `KEY=VALUE`，不要写 YAML 块）。


In [16]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
ROOT = Path(r"d:\myproject\enterprise_bench_langchain")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass

load_dotenv(ROOT / ".env")

print("项目根目录:", ROOT)
print("DASHSCOPE_API_KEY:", "已配置" if os.getenv("DASHSCOPE_API_KEY", "").strip() else "未配置")
print("LLM API:", "已配置" if (os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY") or "").strip() else "未配置")

项目根目录: d:\myproject\enterprise_bench_langchain
DASHSCOPE_API_KEY: 已配置
LLM API: 已配置


## 架构（先建立直觉）

```
用户说的话 ──►【抽取】──►【去重/合并】──► 变成向量 ──► 存进 Chroma
新问题     ──►【向量检索 + BM25】──►【重排】──► 和最近聊天一起 ──► 拼成 Prompt ──► 问大模型
```


## 源码 A：混合检索 + 记忆抽取/合并

完整实现见 **`memory_hybrid_compressor.py`**（`HybridRetriever`、`MemoryCompressor`）。运行下一格 import 后即可使用。


In [ ]:
# 源码 A 已抽到 memory_hybrid_compressor.py，此处直接 import
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path and (Path.cwd() / "memory_hybrid_compressor.py").exists():
    sys.path.insert(0, str(Path.cwd()))
elif str(Path.cwd() / "Chapter-3") not in sys.path:
    c3 = Path.cwd() / "Chapter-3"
    if (c3 / "memory_hybrid_compressor.py").exists():
        sys.path.insert(0, str(c3))

from memory_hybrid_compressor import HybridRetriever, MemoryCompressor

print("✓ 源码 A 已加载：HybridRetriever、MemoryCompressor（见 memory_hybrid_compressor.py）")


## 源码 B：长期记忆系统

完整实现见 **`memory_ltm_improved.py`**（`SelfBuiltLongTermMemoryImproved` 等）。运行下一格 import 后即可使用。


In [ ]:
# 源码 B 已抽到 memory_ltm_improved.py，此处直接 import
from memory_ltm_improved import (
    MEMORY_PROMPT_TEMPLATE,
    MemoryWritePipeline,
    SelfBuiltLongTermMemoryImproved,
    ThreadShortTermMemory,
    WritePipelineConfig,
    close_chroma_ltm,
    default_llm,
    get_or_reset_ltm,
    reset_chroma_directory,
)

print("✓ 源码 B 已加载：SelfBuiltLongTermMemoryImproved 等（见 memory_ltm_improved.py）")


---
## 模块 1 · 记忆抽取

调用源码 A 里的 `MemoryCompressor.extract_memory`。


In [19]:
llm = default_llm()
compressor = MemoryCompressor(llm)

RAW = "用户喜欢住海景房，预算每晚不超过800元"
extracted = compressor.extract_memory(RAW)

print("【原文】", RAW)
print("【抽取结果】")
print(json.dumps({k: extracted[k] for k in ("summary", "key_points", "importance")}, ensure_ascii=False, indent=2))

【原文】 用户喜欢住海景房，预算每晚不超过800元
【抽取结果】
{
  "summary": "用户偏好海景房，预算每晚不超过800元",
  "key_points": [
    "喜欢海景房",
    "预算每晚不超过800元"
  ],
  "importance": 0.9
}


---
## 模块 2 · 记忆合并


In [20]:
old_memory = {
    "id": "mem-old-001",
    "content": "用户喜欢住海景房，预算每晚不超过800元",
    "summary": "用户偏好海景住宿且预算有限",
    "key_points": ["喜欢海景房", "预算800元以内", "每晚"],
    "importance": 0.8,
}
new_memory = compressor.extract_memory("用户说以后订酒店都要海景房，价格别超过800")
merged = compressor.merge_memories([old_memory, new_memory])

print("合并后:", json.dumps({k: merged[k] for k in ("summary", "key_points", "importance")}, ensure_ascii=False, indent=2))

合并后: {
  "summary": "用户今后预订酒店时必须选择海景房，且每晚价格不超过800元。",
  "key_points": [
    "偏好海景房",
    "预算上限为每晚800元",
    "此要求适用于所有未来的酒店预订"
  ],
  "importance": 0.9
}


---
## 模块 3 · 存储

调用 `ingest_long_term` 写入 Chroma。

> **Windows / 重复运行**：默认用 `get_or_reset_ltm` **复用** 同一个 `ltm`，只 `clear_all_memories()`，**不删** `chroma_notebook_demo` 文件夹（避免 WinError 32）。  
> 只有 Restart Kernel 之后，才可将下面 `FORCE_REBUILD_DIRECTORY = True` 尝试整目录重建。


In [21]:
USER_ID = "student_demo"
CHROMA_DIR = ROOT / "chroma_notebook_dem"
FORCE_REBUILD_DIRECTORY = False  # Restart Kernel 后可改 True 尝试删目录重建

ltm = get_or_reset_ltm(
    user_id=USER_ID,
    persist_directory=CHROMA_DIR,
    embedding_model="dashscope",
    collection_name="notebook_long_term_v1",
    similarity_threshold=0.0,
    force_rebuild_directory=FORCE_REBUILD_DIRECTORY,
)

samples = [
    ("用户喜欢住海景房，预算每晚不超过800元", "preference", 0.8),
    ("用户计划下个月去三亚旅游", "fact", 0.5),
    ("用户偏好海景住宿，价格控制在800以内", "preference", None),  # 近似重复 → 合并
]
for text, mtype, imp in samples:
    mid = ltm.ingest_long_term(text, memory_type=mtype, importance=imp)
    print(f"写入 id={mid[:8]}… | {mtype} | {text[:20]}…")
print(f"\n库内条数: {len(ltm.documents)}（期望 2，第3条应合并）")

写入 id=3e4bfa45… | preference | 用户喜欢住海景房，预算每晚不超过800元…
写入 id=315e6e41… | fact | 用户计划下个月去三亚旅游…
写入 id=3e4bfa45… | preference | 用户偏好海景住宿，价格控制在800以内…

库内条数: 2（期望 2，第3条应合并）


In [22]:
print("=== 库内记忆 ===")
for i, (doc_id, doc, meta) in enumerate(zip(ltm.ids, ltm.documents, ltm.metadatas), 1):
    print(f"[{i}] {meta.get('summary')} | type={meta.get('memory_type')} | imp={meta.get('importance')}")

=== 库内记忆 ===
[1] 用户喜欢海景住宿，预算每晚不超过800元。 | type=preference | imp=0.5
[2] 用户计划下个月去三亚旅游 | type=fact | imp=0.5


---
## 模块 4 · 检索

**打分链路（改进后）**：

1. **vector_sim**：Chroma 距离 → `1 - dist/2`（反映 query 与记忆的语义接近程度）
2. **rrf_score**：向量排名 + jieba BM25 排名的 RRF 融合（只看名次，不看距离幅度）
3. **base = rrf_score × vector_sim**：把排名与语义幅度结合
4. **final_score = base × time_decay × (0.5 + importance)**：时间衰减 + 重要性**乘性**加权

下面用两个 query 对比：排序仍由 RRF 主导，但 **final_score 会随 vector_sim 变化**。


In [26]:
def run_search_demo(query: str):
    print("=" * 72)
    print(f"QUERY: {query}")
    dense = ltm.collection.query(query_texts=[query], n_results=5, where={"user_id": USER_ID})
    print("\n【向量检索】")
    for mid, dist, meta in zip(dense["ids"][0], dense["distances"][0], dense["metadatas"][0]):
        sim = SelfBuiltLongTermMemoryImproved._distance_to_similarity(dist)
        print(f"  {meta.get('summary','')[:36]} | dist={dist:.4f} | vector_sim={sim:.4f}")

    hits = ltm.search_memories(query, top_k=5)
    print("\n【混合检索 search_memories — 分项 score】")
    for h in hits:
        print(
            f"  [{h['memory_type']}] {h['summary']}\n"
            f"    vector_sim={h['vector_sim']:.4f}  "
            f"rrf_score={h['rrf_score']:.4f}  "
            f"final_score={h['final_score']:.4f}"
        )
    return hits


hits_hotel = run_search_demo("你还记得我喜欢住什么样的酒店吗？")
print()
QUERY = "你还记得我喜欢住什么样的酒店吗？"
hits = hits_hotel  # 模块 5 默认用酒店 query 的结果

QUERY: 你还记得我喜欢住什么样的酒店吗？

【向量检索】
  用户喜欢海景住宿，预算每晚不超过800元。 | dist=0.4616 | vector_sim=0.7692
  用户计划下个月去三亚旅游 | dist=0.5102 | vector_sim=0.7449

【混合检索 search_memories — 分项 score】
  [preference] 用户喜欢海景住宿，预算每晚不超过800元。
    vector_sim=0.7692  rrf_score=0.0167  final_score=0.0128
  [fact] 用户计划下个月去三亚旅游
    vector_sim=0.7449  rrf_score=0.0164  final_score=0.0122



---
## 模块 5 · 使用（拼 Prompt）


In [27]:
THREAD_ID = "thread_notebook_001"
ltm.record_turn(THREAD_ID, "你好，我想规划一次旅行", "你好！请问您想去哪里旅行呢？")
prompt = ltm.build_prompt(THREAD_ID, QUERY, long_term_hits=hits)
print(prompt)

你是一个有记忆的智能助手。请结合【最近对话】与【相关长期记忆】回答用户问题。

【最近对话】（来自 Checkpoint / 本线程短期缓冲，按时间顺序）
user: 你好，我想规划一次旅行
assistant: 你好！请问您想去哪里旅行呢？
user: 你好，我想规划一次旅行
assistant: 你好！请问您想去哪里旅行呢？

【相关长期记忆】（向量检索 + 重排后的 top-k）
- [preference] 用户喜欢海景住宿，预算每晚不超过800元。 (final=0.0128, vector_sim=0.7692, rrf=0.0167)
- [fact] 用户计划下个月去三亚旅游 (final=0.0122, vector_sim=0.7449, rrf=0.0164)

【当前问题】
你还记得我喜欢住什么样的酒店吗？



In [28]:
RUN_LLM = True  # 改为 True 可让大模型根据 Prompt 回答

if RUN_LLM:
    print(llm.invoke(prompt).content)
else:
    print("RUN_LLM=False：仅展示 Prompt。改 True 可体验完整闭环。")

当然记得！根据您之前提供的信息：

- **住宿偏好**：您喜欢 **海景房**，希望能够欣赏到海边的美景。  
- **预算限制**：每晚的住宿费用 **不超过 800 元**。  

另外，您计划下个月前往 **三亚** 旅游，这正是一个拥有众多海景酒店的热门目的地。如果您需要，我可以帮您挑选几家符合这些条件的酒店，或者提供一些性价比高的海景住宿推荐。请告诉我您还有哪些具体需求（比如入住日期、是否需要早餐、是否偏好度假村或精品酒店等），我会为您进一步细化方案。


---
## 小结

1. **源码 A** 负责「读懂一句话」和「合并两条记忆」。  
2. **源码 B** 负责「存进向量库」「搜出来」「塞进 Prompt」。
4. **抽取**：非结构化对话 → 结构化记忆字段。
5. **合并**：相似记忆归并，控制库膨胀。
6. **存储**：向量库 + 元数据 + BM25 倒排索引同步刷新。
7. **检索**：语义 + 关键词混合，比单一向量更稳。
8. **使用**：短期上下文 + 长期事实一起进 Prompt，才是「有记忆的 Agent」。

